# ⚖️ PakLex AI — Notebook 2: RAG Engine with LangChain + Groq

**Purpose:** Load ChromaDB, retrieve relevant PPC chunks, build a grounded prompt using LangChain, call **Groq API (Mixtral 8x7B)**, and return cited answers.

---
| Step | Task |
|------|------|
| 1 | Load `.env` → Groq API key (secure, never hardcoded) |
| 2 | Load embeddings + ChromaDB vector store |
| 3 | Build LangChain retriever |
| 4 | Define grounding prompt template |
| 5 | Build LangChain RAG chain (LCEL) |
| 6 | Test end-to-end with real queries |

---
> ⚠️ **Prerequisites:**
> - `01_ingest.ipynb` must have been run (vector store must exist)
> - `.env` file must contain `GROQ_API_KEY=gsk_...`

## Step 1 — Load API Key Securely from .env

In [1]:
import os
from dotenv import load_dotenv

# Load .env file — API key stays on your machine, never in code
load_dotenv('../.env')
GROQ_API_KEY = os.getenv('GROQ_API_KEY')

if not GROQ_API_KEY:
    raise ValueError(
        '❌ GROQ_API_KEY missing!\n'
        '   Create ../.env and add: GROQ_API_KEY=gsk_your_key_here\n'
        '   Get free key at: https://console.groq.com'
    )

print(f'✅ Groq API key loaded securely: gsk_...{GROQ_API_KEY[-6:]}')
print('   (Key read from .env — not visible in code or GitHub)')

✅ Groq API key loaded securely: gsk_...Xwtfbg
   (Key read from .env — not visible in code or GitHub)


## Step 2 — Load Embeddings & ChromaDB

In [2]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

VECTOR_STORE_DIR = '../vector_store'
COLLECTION_NAME  = 'pakistan_penal_code'
EMBEDDING_MODEL  = 'all-MiniLM-L6-v2'
TOP_K            = 15

# Load embedding model (same one used during ingestion)
print('⏳ Loading embedding model...')
embeddings = HuggingFaceEmbeddings(
    model_name    = EMBEDDING_MODEL,
    model_kwargs  = {'device': 'cpu'},
    encode_kwargs = {'normalize_embeddings': True},
)
print(f'✅ Embeddings ready: {EMBEDDING_MODEL}')

# Load existing ChromaDB (created by 01_ingest.ipynb)
print('\n⏳ Loading ChromaDB vector store...')
vectordb = Chroma(
    persist_directory = VECTOR_STORE_DIR,
    embedding_function= embeddings,
    collection_name   = COLLECTION_NAME,
)

doc_count = vectordb._collection.count()
print(f'✅ ChromaDB loaded: {doc_count} chunks available')

⏳ Loading embedding model...


C:\Users\Umair_Anjum\AppData\Local\Programs\Python\Python311\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


✅ Embeddings ready: all-MiniLM-L6-v2

⏳ Loading ChromaDB vector store...


C:\Users\Umair_Anjum\AppData\Local\Temp\ipykernel_6152\2274169715.py:20: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 0.4. An updated version of the class exists in the langchain-chroma package and should be used instead. To use it run `pip install -U langchain-chroma` and import as `from langchain_chroma import Chroma`.
  vectordb = Chroma(


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


✅ ChromaDB loaded: 4732 chunks available


## Step 3 — Build LangChain Retriever

In [3]:
# LangChain retriever wraps ChromaDB similarity search
retriever = vectordb.as_retriever(
    search_type   = 'similarity',
    search_kwargs = {'k': TOP_K}
)

# Quick retrieval test
test_docs = retriever.get_relevant_documents('What is punishment for theft?')
print(f'✅ Retriever working — fetched {len(test_docs)} chunks')
print(f'\nTop result preview:')
print(f'   Page     : {test_docs[0].metadata.get("page", "?")}')
print(f'   Source   : {test_docs[0].metadata.get("source_file", "")}')
print(f'   Text     : {test_docs[0].page_content[:200]}...')

C:\Users\Umair_Anjum\AppData\Local\Temp\ipykernel_6152\755533982.py:8: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use invoke instead.
  test_docs = retriever.get_relevant_documents('What is punishment for theft?')
Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


✅ Retriever working — fetched 15 chunks

Top result preview:
   Page     : 136
   Source   : Pak_penal_code.pdf
   Text     : either description for a term which may extend to three y ears, or with fine or with both. 
380. Theft in dwelling house. etc. Whoever commits theft in any building, tent or vessel, 
which building, t...


## Step 4 — Load Groq LLM via LangChain

In [4]:
from langchain_groq import ChatGroq

# ── Groq LLM via LangChain ─────────────────────────────────────────────────
# Model: mixtral-8x7b-32768
# Why Mixtral? Strong legal reasoning, 32K context window, free on Groq

llm = ChatGroq(
    groq_api_key = GROQ_API_KEY,
    model_name   = 'llama-3.1-8b-instant', 
    temperature  = 0.1,     
    max_tokens   = 1024,
)

# Quick test
test_resp = llm.invoke('Say: PakLex AI Ready')
print(f'✅ Groq LLM connected')
print(f'   Model    : mixtral-8x7b-32768')
print(f'   Response : {test_resp.content}')

✅ Groq LLM connected
   Model    : mixtral-8x7b-32768
   Response : PakLex AI Ready. How can I assist you today?


## Step 5 — Define Grounding Prompt Template

In [5]:
from langchain.prompts import ChatPromptTemplate

# ── Strict grounding prompt ────────────────────────────────────────────────
# This prompt forces Groq to answer ONLY from retrieved PPC context.
# It prevents hallucination by explicitly forbidding outside knowledge.

PROMPT_TEMPLATE = """\
You are PakLex AI, an expert legal assistant specialising in the Pakistan Penal Code (PPC) 1860.

STRICT RULES — follow exactly:
1. Answer ONLY using the provided CONTEXT below. Do not use any outside knowledge.
2. Always mention the specific Section number (e.g. Section 302) when referencing the law.
3. If the context does not contain enough information to answer, respond with:
   "I don't have enough information from the Pakistan Penal Code to answer this question."
4. Never speculate, guess, or hallucinate legal provisions.
5. Be clear, precise, and professional in your language.
6. Structure your answer: cite the section first, then explain it.

=== CONTEXT FROM PAKISTAN PENAL CODE 1860 ===
{context}
=== END CONTEXT ===

Question: {question}

Answer:"""

prompt = ChatPromptTemplate.from_template(PROMPT_TEMPLATE)
print('✅ Prompt template defined')
print(f'   Variables: {{context}}, {{question}}')
print(f'   Grounding: strict — answers only from context')

✅ Prompt template defined
   Variables: {context}, {question}
   Grounding: strict — answers only from context


## Step 6 — Build RAG Chain with LangChain LCEL

In [6]:
import time
from langchain.schema.runnable import RunnablePassthrough
from langchain.schema.output_parser import StrOutputParser

def format_docs(docs) -> str:
    """
    Format retrieved documents into a single context string.
    Each chunk is labelled with its page number and source file.
    """
    parts = []
    for i, doc in enumerate(docs):
        page   = doc.metadata.get('page', '?')
        source = doc.metadata.get('source_file', 'PPC')
        parts.append(
            f'[Chunk {i+1} | {source} | Page {page}]\n{doc.page_content}'
        )
    return '\n\n'.join(parts)


# ── LangChain LCEL RAG chain ───────────────────────────────────────────────
# Flow: question → retrieve docs → format → prompt → LLM → parse output
rag_chain = (
    {
        'context' : retriever | format_docs,  # retrieve + format
        'question': RunnablePassthrough(),     # pass question through unchanged
    }
    | prompt          # inject into prompt template
    | llm             # send to Groq Mixtral
    | StrOutputParser()  # extract text from response
)

print('✅ RAG chain built with LangChain LCEL')
print('   Flow: question → ChromaDB → Prompt → Groq Mixtral → Answer')

✅ RAG chain built with LangChain LCEL
   Flow: question → ChromaDB → Prompt → Groq Mixtral → Answer


## Step 7 — Full rag_query() Function (used by Flask app)

In [7]:
def rag_query(query: str) -> dict:
    """
    Full RAG pipeline:
      1. Retrieve top-5 PPC chunks from ChromaDB
      2. Format context
      3. Send to Groq Mixtral via LangChain
      4. Return answer + source metadata + timing

    Returns:
      {
        'answer'       : str,
        'sources'      : list[dict],
        'response_time': float,
        'query'        : str,
      }
    """
    t0 = time.time()

    # 1. Retrieve relevant chunks
    docs = retriever.get_relevant_documents(query)

    if not docs:
        return {
            'answer'       : 'No relevant sections found in the Pakistan Penal Code.',
            'sources'      : [],
            'response_time': round(time.time() - t0, 2),
            'query'        : query,
        }

    # 2. Run through RAG chain
    answer = rag_chain.invoke(query)

    # 3. Build source citations list
    sources = []
    for doc in docs:
        sources.append({
            'content'       : doc.page_content,
            'page'          : doc.metadata.get('page', ''),
            'source_file'   : doc.metadata.get('source_file', ''),
            'section_number': doc.metadata.get('section_number', ''),
            'section_title' : doc.metadata.get('section_title', ''),
            'chapter'       : doc.metadata.get('chapter', ''),
        })

    return {
        'answer'       : answer,
        'sources'      : sources,
        'response_time': round(time.time() - t0, 2),
        'query'        : query,
    }


print('✅ rag_query() function ready')

✅ rag_query() function ready


## Step 8 — Test Queries (Live Demo)

In [8]:
# ── Test 1: Murder ─────────────────────────────────────────────────────────
result = rag_query('What is Qatl-i-amd and what is its punishment under PPC?')

print('=' * 70)
print(f'QUERY   : {result["query"]}')
print(f'TIME    : {result["response_time"]}s')
print('=' * 70)
print(f'\nANSWER:\n{result["answer"]}')
print(f'\nSOURCES ({len(result["sources"])} chunks retrieved):')
for i, s in enumerate(result['sources']):
    print(f'  [{i+1}] Page {s["page"]} — {s["source_file"]}')
    print(f'       {s["content"][:120]}...')

QUERY   : What is Qatl-i-amd and what is its punishment under PPC?
TIME    : 1.19s

ANSWER:
I don't have enough information from the provided context to answer the question about Qatl-i-amd. However, I can provide information about Qatl shibh-i-amd.

Section 315. Qatl shibh­i -amd.__ Whoever, with intent to cause harm to the body or mind of any person , causes the death of that or of any other person by means of a weapon or an act which in the ordinary course of nature is not likely to cause death is said to commit qatl­shibh­i­ ’amd.

Section 316. Punishment for qatl shibh­i­‘amd .__ Whoever commits qatl shibh -i-amd shall be liable to diyat and may also be punished with imprisonment of either description for a term which may extend to 1[twenty five years] as ta’zir.

SOURCES (15 chunks retrieved):
  [1] Page 110 — Pak_penal_code.pdf
       ordinary course of nature is not likely to cause death is said to commit qatl­shibh­i­ ’amd. 
Illustration 
A in order t...
  [2] Page 110 — Pak_p

In [9]:
# ── Test 2: Theft ──────────────────────────────────────────────────────────
result2 = rag_query('What is the punishment for theft and robbery on a highway?')
print('=' * 70)
print(f'QUERY   : {result2["query"]}')
print(f'TIME    : {result2["response_time"]}s')
print('=' * 70)
print(f'\nANSWER:\n{result2["answer"]}')

QUERY   : What is the punishment for theft and robbery on a highway?
TIME    : 49.09s

ANSWER:
Section 392. 

The punishment for robbery is rigorous imprisonment for a term which shall not be less than three years nor more than ten years, and shall also be liable to fine. If the robbery be committed on the highway, the imprisonment may be extended to fourteen years.


In [10]:
# ── Test 3: Out of scope (model should gracefully decline) ─────────────────
result3 = rag_query('What are the inheritance rules for property in Pakistan?')
print('=' * 70)
print(f'QUERY   : {result3["query"]}')
print('=' * 70)
print(f'\nANSWER:\n{result3["answer"]}')
print('\n✓ Expected: model should decline (inheritance = civil law, not PPC)')

QUERY   : What are the inheritance rules for property in Pakistan?

ANSWER:
498A. Whoever by deceitful or illegal means deprives any woman from inheriting any movable or immovable property at the time of opening of succession shall be punished with imprisonment for either description for a term which may extend to ten years but not be less than five years or with a fine of one million rupees or both.

This section prohibits depriving women from inheriting property through deceitful or illegal means. It aims to protect the rights of women in inheriting property and provides a punishment for those who violate this right.

✓ Expected: model should decline (inheritance = civil law, not PPC)


## ✅ RAG Engine Verified!

**Next:** Open `03_app.ipynb` to launch Flask server and connect to the frontend.

```
01_ingest.ipynb    → builds vector_store/  ✓
02_rag_engine.ipynb → RAG pipeline tested  ✓ you are here
03_app.ipynb       → Flask API + frontend  → next
```